In [ ]:
from pathlib import Path

for p in Path("/kaggle/input").iterdir():
    print(p)

In [ ]:
from pathlib import Path

for p in Path("/kaggle/input/notebooks").rglob("*"):
    print(p)

In [ ]:
from pathlib import Path
import pandas as pd

MASTER = Path("/kaggle/input/notebooks/shri7ul/master-dataset-creation/master/master_train.parquet")

master_train = pd.read_parquet(MASTER)
master_train.head()

In [ ]:
# ==========================================================
# Trace The Ace V2
# Stage 02 - Exploratory Data Analysis (EDA)
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import plotly.express as px
import plotly.graph_objects as go

import seaborn as sns

from IPython.display import display

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

MASTER = Path(
    "/kaggle/input/notebooks/shri7ul/master-dataset-creation/master/master_train.parquet"
)

EDA_OUTPUT = Path("/kaggle/working/eda")
EDA_OUTPUT.mkdir(exist_ok=True)

In [ ]:
master = pd.read_parquet(MASTER)

print("=" * 60)
print("MASTER DATASET")
print("=" * 60)

print(f"Rows      : {master.shape[0]:,}")
print(f"Columns   : {master.shape[1]}")

display(master.head())

In [ ]:
master.info()

In [ ]:
master.describe(include="all").T

In [ ]:
missing = (
    master
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing = missing[missing > 0]

display(missing)

if not missing.empty:
    plt.figure(figsize=(10, 4))
    missing.plot.bar()
    plt.title("Missing Values")
    plt.show()
else:
    print("✅ No missing values found.")

In [ ]:
print("Duplicate response_id")

print(master.response_id.duplicated().sum())

print()

print("Duplicate session_id")

print(master.session_id.duplicated().sum())

In [ ]:
health = pd.DataFrame({

    "Rows":[master.shape[0]],

    "Columns":[master.shape[1]],

    "Unique Responses":[master.response_id.nunique()],

    "Unique Sessions":[master.session_id.nunique()],

    "Missing Cells":[master.isna().sum().sum()],

    "Duplicate Response":[master.response_id.duplicated().sum()]

})

display(health)

In [ ]:
# ==========================================================
# Target Distribution
# ==========================================================

target = (
    master["is_correct"]
    .value_counts()
    .sort_index()
)

target_pct = (
    master["is_correct"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

summary = pd.DataFrame({
    "Count": target,
    "Percentage": target_pct.round(2)
})

display(summary)

plt.figure(figsize=(6,4))

plt.bar(
    ["Incorrect (0)", "Correct (1)"],
    target.values
)

plt.title("Target Distribution")

plt.ylabel("Number of Responses")

plt.show()

In [ ]:
positive = master["is_correct"].mean()

print(f"Positive Rate : {positive:.4f}")

print(f"Negative Rate : {1-positive:.4f}")

In [ ]:
# ==========================================================
# Learning Objective Overview
# ==========================================================

print("=" * 60)
print("Learning Objective Statistics")
print("=" * 60)

print(f"Unique Learning Objectives : {master.learning_objective.nunique()}")
print(f"Unique Objective IDs       : {master.learning_objective_id.nunique()}")

In [ ]:
# ==========================================================
# Top Learning Objectives
# ==========================================================

top_objectives = (
    master["learning_objective"]
    .value_counts()
    .head(20)
)

display(top_objectives)

plt.figure(figsize=(12,7))

top_objectives.sort_values().plot(kind="barh")

plt.title("Top 20 Most Frequent Learning Objectives")

plt.xlabel("Number of Responses")

plt.show()

In [ ]:
objective_freq = (
    master["learning_objective"]
    .value_counts()
)

plt.figure(figsize=(8,5))

plt.hist(
    objective_freq,
    bins=40
)

plt.title("Distribution of Responses per Learning Objective")

plt.xlabel("Responses")

plt.ylabel("Number of Objectives")

plt.show()

In [ ]:
rare = objective_freq[objective_freq <= 10]

print("=" * 60)

print(f"Objectives with <=10 samples : {len(rare)}")

print("=" * 60)

display(rare.head(20))

In [ ]:
objective_success = (
    master
    .groupby("learning_objective")
    ["is_correct"]
    .agg(["count","mean"])
    .sort_values("mean", ascending=False)
)

display(objective_success.head(20))

In [ ]:
hard = (
    objective_success
    .query("count >= 30")
    .sort_values("mean")
    .head(20)
)

display(hard)

plt.figure(figsize=(12,7))

hard["mean"].sort_values().plot(kind="barh")

plt.title("Hardest Learning Objectives")

plt.xlabel("Correct Rate")

plt.show()

In [ ]:
easy = (
    objective_success
    .query("count >= 30")
    .sort_values("mean", ascending=False)
    .head(20)
)

display(easy)

plt.figure(figsize=(12,7))

easy["mean"].sort_values().plot(kind="barh")

plt.title("Easiest Learning Objectives")

plt.xlabel("Correct Rate")

plt.show()

## Observation

- Dataset contains 398 unique learning objectives.
- Distribution is highly imbalanced with a long-tail pattern.
- 204 objectives have 10 or fewer training samples.
- Objective difficulty varies substantially; some objectives have >90% correctness while others are below 25%.

## Evidence

- Long-tail histogram of objective frequencies.
- Objective-wise success rate ranges approximately from 0.24 to 0.95.

## Hypothesis

Learning objective carries meaningful predictive information. Both objective popularity and semantic meaning are likely correlated with student success. Rare objectives may generalize differently from common ones.

## Experiments to Test

- Add objective frequency encoding.
- Add rare objective indicator.
- Compare Transcript vs Objective vs Transcript+Objective.
- Use objective sentence embeddings.
- Evaluate fold-wise target encoding for objective IDs.

In [ ]:
# ==========================================================
# Transcript Statistics
# ==========================================================

master["transcript_chars"] = master["transcript"].str.len()

master["student_chars"] = master["student_text"].str.len()

master["tutor_chars"] = master["tutor_text"].str.len()

master["background_chars"] = master["background_text"].str.len()

stats = master[
    [
        "transcript_chars",
        "student_chars",
        "tutor_chars",
        "background_chars",
    ]
].describe().T

display(stats)

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    master["transcript_chars"],
    bins=60
)

plt.title("Transcript Length Distribution")

plt.xlabel("Characters")

plt.ylabel("Sessions")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    master["student_chars"],
    master["tutor_chars"],
    alpha=0.2,
    s=8
)

plt.xlabel("Student Characters")

plt.ylabel("Tutor Characters")

plt.title("Student vs Tutor Text Length")

plt.show()

In [ ]:
master["tutor_student_ratio"] = (
    master["tutor_chars"] + 1
) / (
    master["student_chars"] + 1
)

master["tutor_student_ratio"].describe()

In [ ]:
plt.figure(figsize=(9,5))

plt.hist(
    master["tutor_student_ratio"],
    bins=60
)

plt.title("Tutor / Student Character Ratio")

plt.xlabel("Ratio")

plt.show()

In [ ]:
master["background_ratio"] = (
    master["background_chars"]
    /
    master["transcript_chars"]
)

master["background_ratio"].describe()

In [ ]:
plt.figure(figsize=(9,5))

plt.hist(
    master["background_ratio"],
    bins=40
)

plt.title("Background Text Ratio")

plt.xlabel("Background Ratio")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="transcript_chars",
    by="is_correct",
    grid=False
)

plt.title("Transcript Length vs Target")

plt.suptitle("")

plt.xlabel("Correct")

plt.ylabel("Transcript Characters")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="tutor_chars",
    by="is_correct",
    grid=False
)

plt.title("Tutor Text Length vs Target")

plt.suptitle("")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="student_chars",
    by="is_correct",
    grid=False
)

plt.title("Student Text Length vs Target")

plt.suptitle("")

plt.show()

In [ ]:
# ==========================================================
# Conversation Structure
# ==========================================================

master["student_turns"] = master["transcript"].str.count(r"\[STUDENT\]")

master["tutor_turns"] = master["transcript"].str.count(r"\[TUTOR\]")

master["background_turns"] = master["transcript"].str.count(r"\[BACKGROUND\]")

master["total_turns"] = (
    master["student_turns"]
    + master["tutor_turns"]
    + master["background_turns"]
)

display(
    master[
        [
            "student_turns",
            "tutor_turns",
            "background_turns",
            "total_turns",
        ]
    ].describe().T
)

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    master["total_turns"],
    bins=60
)

plt.title("Total Conversation Turns")

plt.xlabel("Turns")

plt.ylabel("Sessions")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="total_turns",
    by="is_correct",
    grid=False
)

plt.title("Total Turns vs Target")

plt.suptitle("")

plt.show()

In [ ]:
master["turn_ratio"] = (
    master["tutor_turns"] + 1
) / (
    master["student_turns"] + 1
)

master["turn_ratio"].describe()

In [ ]:
master["turn_ratio"] = (
    master["tutor_turns"] + 1
) / (
    master["student_turns"] + 1
)

master["turn_ratio"].describe()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="turn_ratio",
    by="is_correct",
    grid=False
)

plt.title("Tutor / Student Turn Ratio vs Target")

plt.suptitle("")

plt.show()

In [ ]:
master["student_turn_fraction"] = (
    master["student_turns"] /
    master["total_turns"]
)

master["tutor_turn_fraction"] = (
    master["tutor_turns"] /
    master["total_turns"]
)

master["background_turn_fraction"] = (
    master["background_turns"] /
    master["total_turns"]
)

display(
    master[
        [
            "student_turn_fraction",
            "tutor_turn_fraction",
            "background_turn_fraction"
        ]
    ].describe().T
)

In [ ]:
# ==========================================================
# Question Statistics
# ==========================================================

master["question_marks"] = master["transcript"].str.count(r"\?")

master["student_questions"] = master["student_text"].str.count(r"\?")

master["tutor_questions"] = master["tutor_text"].str.count(r"\?")

display(
    master[
        [
            "question_marks",
            "student_questions",
            "tutor_questions",
        ]
    ].describe().T
)

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    master["question_marks"],
    bins=50
)

plt.title("Question Marks Distribution")

plt.xlabel("Number of Questions")

plt.ylabel("Sessions")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="question_marks",
    by="is_correct",
    grid=False
)

plt.title("Questions vs Target")

plt.suptitle("")

plt.show()

In [ ]:
master["unclear_count"] = (
    master["transcript"]
    .str.lower()
    .str.count(r"\[unclear\]|unclear")
)

display(
    master["unclear_count"].describe()
)

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    master["unclear_count"],
    bins=40
)

plt.title("Unclear Count Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

master.boxplot(
    column="unclear_count",
    by="is_correct",
    grid=False
)

plt.title("Unclear Count vs Target")

plt.suptitle("")

plt.show()

In [ ]:
positive_words = [
    "good",
    "great",
    "excellent",
    "awesome",
    "perfect",
    "correct",
    "nice",
    "well done",
    "brilliant"
]

for word in positive_words:
    master[f"word_{word.replace(' ','_')}"] = (
        master["tutor_text"]
        .str.lower()
        .str.count(word)
    )

display(
    master[
        [c for c in master.columns if c.startswith("word_")]
    ].describe().T
)

In [ ]:
behavior_words = [
    "why",
    "because",
    "explain",
    "think",
    "remember",
    "try",
    "first",
    "next",
    "good",
    "great",
    "excellent",
    "correct",
    "okay",
    "well done"
]

for word in behavior_words:
    master[f"tutor_{word.replace(' ','_')}"] = (
        master["tutor_text"]
        .str.lower()
        .str.count(word)
    )

display(
    master[
        [c for c in master.columns if c.startswith("tutor_")]
    ].describe().T
)

In [ ]:
existing_cols = [c for c in numeric_cols if c in master.columns]
print("Existing columns:")
print(existing_cols)

In [ ]:
print(master.columns.tolist())